In [ ]:
#Plot creation for one component (all timesteps)
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# Define the elliptical ROI parameters
def create_ellipse_mask(image, center, axes, angle):
    x, y = np.meshgrid(np.arange(image.shape[1]), np.arange(image.shape[0]))
    yc, xc = center
    a, b = axes
    cos_angle = np.cos(np.radians(angle))
    sin_angle = np.sin(np.radians(angle))
    ellipse = (((x - xc) * cos_angle + (y - yc) * sin_angle) ** 2) / a ** 2 + \
              (((x - xc) * sin_angle - (y - yc) * cos_angle) ** 2) / b ** 2
    mask = ellipse <= 1
    return mask

def display_image_with_ellipse(image, center, axes, angle, timestep):
    # Create an ellipse mask
    mask = create_ellipse_mask(image, center, axes, angle)
    
    # Plot the image and overlay the ellipse
    fig, ax = plt.subplots()
    cax = ax.imshow(image, cmap='gray', origin='lower')
    ax.set_title(f"Slice with Elliptical ROI (Timestep {timestep})")
    ax.set_xlabel("X-axis (pixels)")
    ax.set_ylabel("Y-axis (pixels)")
    
    # Add a color bar for reference
    fig.colorbar(cax, ax=ax, orientation='vertical', label='Intensity')
    
    # Add the ellipse outline
    ellipse = Ellipse(
        xy=center[::-1], width=2 * axes[0], height=2 * axes[1], 
        angle=angle, edgecolor='red', facecolor='none', linewidth=2
    )
    ax.add_patch(ellipse)
    
    # Overlay the mask
    ax.imshow(mask, cmap='jet', alpha=0.4, origin='lower')
    plt.show()

def load_h5_and_draw_ellipse(h5_file_path):
    # Load the HDF5 file
    with h5py.File(h5_file_path, "r") as f:
        v = f["u"][:]  # Assuming the dataset key is 'v'
        num_timesteps = v.shape[0]

        # Define ellipse parameters
        center = (44, 33)  # (y, x) format
        axes = (10, 7)  # (semi-major axis, semi-minor axis)
        angle = 0  # Rotation angle in degrees (ellipse is vertical)

        mean_values = []  # Store mean values for each timestep

        # Loop through all timesteps
        for t in range(num_timesteps):
            slice_v = v[t, :, :, :]  # Current time index
            axial_slice_v = slice_v[:, :, 78]  # Axial slice ..

            # Create an ellipse mask
            mask = create_ellipse_mask(axial_slice_v, center, axes, angle)

            # Display the slice with the elliptical ROI and timestep
            display_image_with_ellipse(axial_slice_v, center, axes, angle, timestep=t)

            # Calculate the mean value of the selected pixels
            selected_pixels = axial_slice_v[mask]
            mean_value = selected_pixels.mean()
            mean_values.append(mean_value)

        # Convert mean values to a NumPy array
        mean_values = np.array(mean_values)

        # Plot the mean values over timesteps
        plt.figure()
        plt.plot(range(num_timesteps), mean_values, marker='o', linestyle='-', color='b')
        plt.title("Mean Pixel Values in ROI Over Timesteps")
        plt.xlabel("Timestep")
        plt.ylabel("Mean Pixel Value")
        plt.grid()
        plt.show()

        print("\nMean values for each timestep:")
        print(mean_values)

# Path to the HDF5 file
h5_file_path = r"Your Path\mri_data.h5"
load_h5_and_draw_ellipse(h5_file_path)

In [ ]:
#Plot creation for all velocity components (all timesteps)
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# Define the elliptical ROI parameters
def create_ellipse_mask(image, center, axes, angle):
    x, y = np.meshgrid(np.arange(image.shape[1]), np.arange(image.shape[0]))
    yc, xc = center
    a, b = axes
    cos_angle = np.cos(np.radians(angle))
    sin_angle = np.sin(np.radians(angle))
    ellipse = (((x - xc) * cos_angle + (y - yc) * sin_angle) ** 2) / a ** 2 + \
              (((x - xc) * sin_angle - (y - yc) * cos_angle) ** 2) / b ** 2
    mask = ellipse <= 1
    return mask

def display_image_with_ellipse(image, center, axes, angle, timestep, component):
    # Create an ellipse mask
    mask = create_ellipse_mask(image, center, axes, angle)
    
    # Plot the image and overlay the ellipse
    fig, ax = plt.subplots()
    cax = ax.imshow(image, cmap='gray', origin='lower')
    ax.set_title(f"{component}-Component: Slice with Elliptical ROI (Timestep {timestep})")
    ax.set_xlabel("X-axis (pixels)")
    ax.set_ylabel("Y-axis (pixels)")
    
    # Add a color bar for reference
    fig.colorbar(cax, ax=ax, orientation='vertical', label='Intensity')
    
    # Add the ellipse outline
    ellipse = Ellipse(
        xy=center[::-1], width=2 * axes[0], height=2 * axes[1], 
        angle=angle, edgecolor='red', facecolor='none', linewidth=2
    )
    ax.add_patch(ellipse)
    
    # Overlay the mask
    ax.imshow(mask, cmap='jet', alpha=0.4, origin='lower')
    plt.show()

def load_h5_and_draw_ellipse(h5_file_path):
    # Load the HDF5 file
    with h5py.File(h5_file_path, "r") as f:
        components = {"u": f["u"][:], "v": f["v"][:], "w": f["w"][:]}  # Assuming the dataset keys are 'u', 'v', 'w'

        # Define ellipse parameters
        center = (44, 33)  # (y, x) format
        axes = (10, 7)  # (semi-major axis, semi-minor axis)
        angle = 0  # Rotation angle in degrees (ellipse is vertical)
        
        # Process each component
        for component_name, component_data in components.items():
            num_timesteps = component_data.shape[0]
            mean_values = []  # Store mean values for each timestep

            # Loop through all timesteps
            for t in range(num_timesteps):
                slice_component = component_data[t, :, :, :]  # Current time index
                axial_slice_component = slice_component[:, :, 78]  # Axial slice (what you choose)

                # Create an ellipse mask
                mask = create_ellipse_mask(axial_slice_component, center, axes, angle)

                # Display the slice with the elliptical ROI and timestep
                display_image_with_ellipse(axial_slice_component, center, axes, angle, timestep=t, component=component_name)

                # Calculate the mean value of the selected pixels
                selected_pixels = axial_slice_component[mask]
                mean_value = selected_pixels.mean()
                mean_values.append(mean_value)

            # Convert mean values to a NumPy array
            mean_values = np.array(mean_values)

            # Plot the mean values over timesteps
            plt.figure()
            plt.plot(range(num_timesteps), mean_values, marker='o', linestyle='-', color='b')
            plt.title(f"Mean Pixel Values in ROI Over Timesteps ({component_name}-Component)")
            plt.xlabel("Timestep")
            plt.ylabel("Mean Pixel Value")
            plt.grid()
            plt.show()

            print(f"\nMean values for each timestep ({component_name}-Component):")
            print(mean_values)

# Path to the HDF5 file
h5_file_path = r"Your Path\mri_data.h5"
load_h5_and_draw_ellipse(h5_file_path)

In [ ]:
#Coordinate system transformation to match the simulation
import h5py
import numpy as np
import matplotlib.pyplot as plt

# Define the elliptical ROI parameters
def create_ellipse_mask(image, center, axes, angle):
    x, y = np.meshgrid(np.arange(image.shape[1]), np.arange(image.shape[0]))
    yc, xc = center
    a, b = axes
    cos_angle = np.cos(np.radians(angle))
    sin_angle = np.sin(np.radians(angle))
    ellipse = (((x - xc) * cos_angle + (y - yc) * sin_angle) ** 2) / a ** 2 + \
              (((x - xc) * sin_angle - (y - yc) * cos_angle) ** 2) / b ** 2
    mask = ellipse <= 1
    return mask

def load_h5_and_process_components(h5_file_path):
    # Load the HDF5 file
    with h5py.File(h5_file_path, "r") as f:
        components = {"u": f["u"][:], "v": f["v"][:], "w": f["w"][:]}  # Assuming the dataset keys are 'u', 'v', 'w'

        # Define ellipse parameters
        center = (44, 33)  # (y, x) format
        axes = (10, 7)  # (semi-major axis, semi-minor axis)
        angle = 0  # Rotation angle in degrees (ellipse is vertical)

        # Initialize dictionaries to store mean values
        mean_values_dict = {"u": [], "v": [], "w": []}

        # Process each component
        for component_name, component_data in components.items():
            num_timesteps = component_data.shape[0]

            # Loop through all timesteps
            for t in range(num_timesteps):
                slice_component = component_data[t, :, :, :]  # Current time index
                axial_slice_component = slice_component[:, :, 78]  #slice selection

                # Create an ellipse mask
                mask = create_ellipse_mask(axial_slice_component, center, axes, angle)

                # Calculate the mean value of the selected pixels
                selected_pixels = axial_slice_component[mask]
                mean_value = selected_pixels.mean()
                mean_values_dict[component_name].append(mean_value)

        # Convert mean values to NumPy arrays
        for key in mean_values_dict:
            mean_values_dict[key] = np.array(mean_values_dict[key])

        # Print the mean values
        for component_name, mean_values in mean_values_dict.items():
            print(f"\nMean values for each timestep ({component_name}-Component):")
            print(mean_values)

        # Apply the transformation to the mean values
        mean_u_new = mean_values_dict["w"]
        mean_v_new = mean_values_dict["u"]
        mean_w_new = -mean_values_dict["v"]

        # Print the transformed mean values
        print("\nTransformed Mean Values:")
        print("Mean values for each timestep (u-Component, transformed):")
        print(mean_u_new)
        print("Mean values for each timestep (v-Component, transformed):")
        print(mean_v_new)
        print("Mean values for each timestep (w-Component, transformed):")
        print(mean_w_new)

        # Plot the original and transformed mean values
        timesteps = np.arange(len(mean_values_dict["u"]))

        # Plot original components
        plt.figure(figsize=(12, 6))
        plt.plot(timesteps, mean_values_dict["u"], label="Original u", marker='o')
        plt.plot(timesteps, mean_values_dict["v"], label="Original v", marker='o')
        plt.plot(timesteps, mean_values_dict["w"], label="Original w", marker='o')
        plt.title("Original Mean Values of Velocity Components")
        plt.xlabel("Timesteps")
        plt.ylabel("Mean Values")
        plt.legend()
        plt.grid()
        plt.show()

        # Plot transformed components
        plt.figure(figsize=(12, 6))
        plt.plot(timesteps, mean_u_new, label="Transformed u", marker='o')
        plt.plot(timesteps, mean_v_new, label="Transformed v", marker='o')
        plt.plot(timesteps, mean_w_new, label="Transformed w", marker='o')
        plt.title("Transformed Mean Values of Velocity Components")
        plt.xlabel("Timesteps")
        plt.ylabel("Mean Values")
        plt.legend()
        plt.grid()
        plt.show()

# Path to the HDF5 file
h5_file_path = r"Your Path\mri_data.h5"
load_h5_and_process_components(h5_file_path)

In [ ]:
#Velocity component data store in txt (patient)
import h5py
import numpy as np
import matplotlib.pyplot as plt

# Define the elliptical ROI parameters
def create_ellipse_mask(image, center, axes, angle):
    x, y = np.meshgrid(np.arange(image.shape[1]), np.arange(image.shape[0]))
    yc, xc = center
    a, b = axes
    cos_angle = np.cos(np.radians(angle))
    sin_angle = np.sin(np.radians(angle))
    ellipse = (((x - xc) * cos_angle + (y - yc) * sin_angle) ** 2) / a ** 2 + \
              (((x - xc) * sin_angle - (y - yc) * cos_angle) ** 2) / b ** 2
    mask = ellipse <= 1
    return mask

def save_transformed_mean_values_to_txt(mean_u_new, mean_v_new, mean_w_new, output_file_path):
    with open(output_file_path, "w") as f:
        #f.write("Transformed Mean Values for each timestep:\n")
        f.write("Timestep\tTransformed u\tTransformed v\tTransformed w\n")
        
        for i, (u, v, w) in enumerate(zip(mean_u_new, mean_v_new, mean_w_new)):
            f.write(f"{i+1}\t{u}\t{v}\t{w}\n")

def load_h5_and_process_components(h5_file_path, output_file_path):
    # Load the HDF5 file
    with h5py.File(h5_file_path, "r") as f:
        components = {"u": f["u"][:], "v": f["v"][:], "w": f["w"][:]}  # Assuming the dataset keys are 'u', 'v', 'w'

        # Define ellipse parameters
        center = (44, 33)  # (y, x) format
        axes = (10, 7)  # (semi-major axis, semi-minor axis)
        angle = 0  # Rotation angle in degrees (ellipse is vertical)

        # Initialize dictionaries to store mean values
        mean_values_dict = {"u": [], "v": [], "w": []}

        # Process each component
        for component_name, component_data in components.items():
            num_timesteps = component_data.shape[0]

            # Loop through all timesteps
            for t in range(num_timesteps):
                slice_component = component_data[t, :, :, :]  # Current time index
                axial_slice_component = slice_component[:, :, 78]  # Axial slice ....

                # Create an ellipse mask
                mask = create_ellipse_mask(axial_slice_component, center, axes, angle)

                # Calculate the mean value of the selected pixels
                selected_pixels = axial_slice_component[mask]
                mean_value = selected_pixels.mean()
                mean_values_dict[component_name].append(mean_value)

        # Convert mean values to NumPy arrays
        for key in mean_values_dict:
            mean_values_dict[key] = np.array(mean_values_dict[key])

        # Apply the transformation to the mean values
        mean_u_new = mean_values_dict["w"]
        mean_v_new = mean_values_dict["u"]
        mean_w_new = -mean_values_dict["v"]

        # Print the transformed mean values
        print("\nTransformed Mean Values:")
        print("Mean values for each timestep (u-Component, transformed):")
        print(mean_u_new)
        print("Mean values for each timestep (v-Component, transformed):")
        print(mean_v_new)
        print("Mean values for each timestep (w-Component, transformed):")
        print(mean_w_new)

        # Save the transformed mean values to a text file
        save_transformed_mean_values_to_txt(mean_u_new, mean_v_new, mean_w_new, output_file_path)

        # Plot the original and transformed mean values
        timesteps = np.arange(len(mean_values_dict["u"]))

        # Plot original components
        plt.figure(figsize=(12, 6))
        plt.plot(timesteps, mean_values_dict["u"], label="Original u", marker='o')
        plt.plot(timesteps, mean_values_dict["v"], label="Original v", marker='o')
        plt.plot(timesteps, mean_values_dict["w"], label="Original w", marker='o')
        plt.title("Original Mean Values of Velocity Components")
        plt.xlabel("Timesteps")
        plt.ylabel("Mean Values")
        plt.legend()
        plt.grid()
        plt.show()

        # Plot transformed components
        plt.figure(figsize=(12, 6))
        plt.plot(timesteps, mean_u_new, label="Transformed u", marker='o')
        plt.plot(timesteps, mean_v_new, label="Transformed v", marker='o')
        plt.plot(timesteps, mean_w_new, label="Transformed w", marker='o')
        plt.title("Transformed Mean Values of Velocity Components")
        plt.xlabel("Timesteps")
        plt.ylabel("Mean Values")
        plt.legend()
        plt.grid()
        plt.show()

# Path to the HDF5 file
h5_file_path = r"Your Path\mri_data.h5"

# Path to save the text file
output_file_path = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_Patient.txt"

load_h5_and_process_components(h5_file_path, output_file_path)

In [ ]:
#Mean value extraction from the cfd array
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# Define the elliptical ROI parameters
def create_ellipse_mask(image, center, axes, angle):
    x, y = np.meshgrid(np.arange(image.shape[1]), np.arange(image.shape[0]))
    yc, xc = center
    a, b = axes
    cos_angle = np.cos(np.radians(angle))
    sin_angle = np.sin(np.radians(angle))
    ellipse = (((x - xc) * cos_angle + (y - yc) * sin_angle) ** 2) / a ** 2 + \
              (((x - xc) * sin_angle - (y - yc) * cos_angle) ** 2) / b ** 2
    mask = ellipse <= 1
    return mask

def display_image_with_ellipse(image, center, axes, angle):
    # Create an ellipse mask
    mask = create_ellipse_mask(image, center, axes, angle)

    # Plot the image and overlay the ellipse
    fig, ax = plt.subplots()
    cax = ax.imshow(image, cmap='gray', origin='lower')
    ax.set_title("Slice with Elliptical ROI")
    ax.set_xlabel("X-axis (pixels)")
    ax.set_ylabel("Y-axis (pixels)")

    # Add a color bar for reference
    fig.colorbar(cax, ax=ax, orientation='vertical', label='Intensity')

    # Add the ellipse outline
    ellipse = Ellipse(
        xy=center[::-1], width=2 * axes[0], height=2 * axes[1], 
        angle=angle, edgecolor='red', facecolor='none', linewidth=2
    )
    ax.add_patch(ellipse)

    # Overlay the mask
    ax.imshow(mask, cmap='jet', alpha=0.4, origin='lower')
    plt.show()

def load_h5_and_process_components(h5_file_path):
    # Load the HDF5 file
    with h5py.File(h5_file_path, "r") as f:
        components = {"u": f["u"][:], "v": f["v"][:], "w": f["w"][:]}  # Assuming the dataset keys are 'u', 'v', 'w'

        # Define ellipse parameters
        center = (13.5, 79)  # (y, x) format
        axes = (4.5, 1.5)  # (semi-major axis, semi-minor axis)
        angle = 0  # Rotation angle in degrees (ellipse is vertical)

        # Initialize dictionaries to store mean values for all timesteps
        mean_values_dict = {"u": [], "v": [], "w": []}

        # Process each component for all 99 timesteps and axial slice ...
        for timestep in range(99):
            for component_name, component_data in components.items():
                slice_component = component_data[timestep, :, :, :]  # Time slice
                axial_slice_component = slice_component[:, 41, :]  # Axial slice ...

                # Create an ellipse mask
                mask = create_ellipse_mask(axial_slice_component, center, axes, angle)

                # Display the slice with the elliptical ROI for the first timestep
                if timestep == 0 and component_name == "v":  # Display only once to avoid redundancy
                    display_image_with_ellipse(axial_slice_component, center, axes, angle)

                # Calculate the mean value of the selected pixels
                selected_pixels = axial_slice_component[mask]
                mean_value = selected_pixels.mean()
                mean_values_dict[component_name].append(mean_value)

        # Convert mean values to NumPy arrays
        for key in mean_values_dict:
            mean_values_dict[key] = np.array(mean_values_dict[key])

        # Print the mean values for all timesteps
        for component_name, mean_values in mean_values_dict.items():
            print(f"\nMean values for all timesteps, axial slice (choosen) ({component_name}-Component):")
            print(mean_values)

# Path to the HDF5 file
h5_file_path = r"Your Path\Step_2_Downsampling\downsampled_simv_v3.h5"
load_h5_and_process_components(h5_file_path)

In [ ]:
#CFD array velocity components plots
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# Define the elliptical ROI parameters
def create_ellipse_mask(image, center, axes, angle):
    x, y = np.meshgrid(np.arange(image.shape[1]), np.arange(image.shape[0]))
    yc, xc = center
    a, b = axes
    cos_angle = np.cos(np.radians(angle))
    sin_angle = np.sin(np.radians(angle))
    ellipse = (((x - xc) * cos_angle + (y - yc) * sin_angle) ** 2) / a ** 2 + \
              (((x - xc) * sin_angle - (y - yc) * cos_angle) ** 2) / b ** 2
    mask = ellipse <= 1
    return mask

def display_image_with_ellipse(image, center, axes, angle):
    # Create an ellipse mask
    mask = create_ellipse_mask(image, center, axes, angle)

    # Plot the image and overlay the ellipse
    fig, ax = plt.subplots()
    cax = ax.imshow(image, cmap='gray', origin='lower')
    ax.set_title("Slice with Elliptical ROI")
    ax.set_xlabel("X-axis (pixels)")
    ax.set_ylabel("Y-axis (pixels)")

    # Add a color bar for reference
    fig.colorbar(cax, ax=ax, orientation='vertical', label='Intensity')

    # Add the ellipse outline
    ellipse = Ellipse(
        xy=center[::-1], width=2 * axes[0], height=2 * axes[1], 
        angle=angle, edgecolor='red', facecolor='none', linewidth=2
    )
    ax.add_patch(ellipse)

    # Overlay the mask
    ax.imshow(mask, cmap='jet', alpha=0.4, origin='lower')
    plt.show()

def load_h5_and_process_components(h5_file_path):
    # Load the HDF5 file
    with h5py.File(h5_file_path, "r") as f:
        components = {"u": f["u"][:], "v": f["v"][:], "w": f["w"][:]}  # Assuming the dataset keys are 'u', 'v', 'w'

        # Define ellipse parameters
        center = (13.5, 79)  # (y, x) format
        axes = (4.5, 1.5)  # (semi-major axis, semi-minor axis)
        angle = 0  # Rotation angle in degrees (ellipse is vertical)

        # Initialize dictionaries to store mean values for all timesteps
        mean_values_dict = {"u": [], "v": [], "w": []}

        # Process each component for all 99 timesteps and axial slice ...
        for timestep in range(99):
            for component_name, component_data in components.items():
                slice_component = component_data[timestep, :, :, :]  # Time slice
                axial_slice_component = slice_component[:, 41, :]  # Axial slice ....

                # Create an ellipse mask
                mask = create_ellipse_mask(axial_slice_component, center, axes, angle)

                # Display the slice with the elliptical ROI for the first timestep
                if timestep == 0 and component_name == "v":  # Display only once to avoid redundancy
                    display_image_with_ellipse(axial_slice_component, center, axes, angle)

                # Calculate the mean value of the selected pixels
                selected_pixels = axial_slice_component[mask]
                mean_value = selected_pixels.mean()
                mean_values_dict[component_name].append(mean_value)

        # Convert mean values to NumPy arrays
        for key in mean_values_dict:
            mean_values_dict[key] = np.array(mean_values_dict[key])

        # Print the mean values for all timesteps
        for component_name, mean_values in mean_values_dict.items():
            print(f"\nMean values for all timesteps, axial slice 64 ({component_name}-Component):")
            print(mean_values)

            # Plot the mean values for each component
            plt.figure(figsize=(10, 6))
            plt.plot(mean_values, label=f"{component_name}-Component", linestyle='-', marker='o')
            plt.title(f"Mean Values for {component_name}-Component Over Time")
            plt.xlabel("Time (Timestep)")
            plt.ylabel("Mean Value")
            plt.legend()
            plt.grid(True)
            plt.show()

# Path to the HDF5 file
h5_file_path = r"Your Path\Step_2_Downsampling\downsampled_simv_v3.h5"
load_h5_and_process_components(h5_file_path)

In [ ]:
#Velocity component values of cfd store in txt file
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# Define the elliptical ROI parameters
def create_ellipse_mask(image, center, axes, angle):
    x, y = np.meshgrid(np.arange(image.shape[1]), np.arange(image.shape[0]))
    yc, xc = center
    a, b = axes
    cos_angle = np.cos(np.radians(angle))
    sin_angle = np.sin(np.radians(angle))
    ellipse = (((x - xc) * cos_angle + (y - yc) * sin_angle) ** 2) / a ** 2 + \
              (((x - xc) * sin_angle - (y - yc) * cos_angle) ** 2) / b ** 2
    mask = ellipse <= 1
    return mask

def display_image_with_ellipse(image, center, axes, angle):
    # Create an ellipse mask
    mask = create_ellipse_mask(image, center, axes, angle)

    # Plot the image and overlay the ellipse
    fig, ax = plt.subplots()
    cax = ax.imshow(image, cmap='gray', origin='lower')
    ax.set_title("Slice with Elliptical ROI")
    ax.set_xlabel("X-axis (pixels)")
    ax.set_ylabel("Y-axis (pixels)")

    # Add a color bar for reference
    fig.colorbar(cax, ax=ax, orientation='vertical', label='Intensity')

    # Add the ellipse outline
    ellipse = Ellipse(
        xy=center[::-1], width=2 * axes[0], height=2 * axes[1], 
        angle=angle, edgecolor='red', facecolor='none', linewidth=2
    )
    ax.add_patch(ellipse)

    # Overlay the mask
    ax.imshow(mask, cmap='jet', alpha=0.4, origin='lower')
    plt.show()

def load_h5_and_process_components(h5_file_path, output_file_path):
    # Load the HDF5 file
    with h5py.File(h5_file_path, "r") as f:
        components = {"u": f["u"][:], "v": f["v"][:], "w": f["w"][:]}  # Assuming the dataset keys are 'u', 'v', 'w'

        # Define ellipse parameters
        center = (13.5, 79)  # (y, x) format
        axes = (4.5, 1.5)  # (semi-major axis, semi-minor axis)
        angle = 0  # Rotation angle in degrees (ellipse is vertical)

        # Open the output file for writing
        with open(output_file_path, 'w') as f_out:
            # Write the header
            f_out.write("Timestep\tTransformed u\tTransformed v\tTransformed w\n")

            # Process each component for all 99 timesteps and axial slice ...
            for timestep in range(99):
                transformed_values = []
                for component_name, component_data in components.items():
                    slice_component = component_data[timestep, :, :, :]  # Time slice
                    axial_slice_component = slice_component[:, 41, :]  # Axial slice ...

                    # Create an ellipse mask
                    mask = create_ellipse_mask(axial_slice_component, center, axes, angle)

                    # Calculate the mean value of the selected pixels
                    selected_pixels = axial_slice_component[mask]
                    mean_value = selected_pixels.mean()
                    transformed_values.append(mean_value)

                # Write the timestep and transformed values to the file
                f_out.write(f"{timestep+1}\t{transformed_values[0]:.15f}\t{transformed_values[1]:.15f}\t{transformed_values[2]:.15f}\n")

        print(f"Mean values have been written to: {output_file_path}")

# Path to the HDF5 file and output text file
h5_file_path = r"Your Path\Step_2_Downsampling\downsampled_simv_v3.h5"
output_file_path = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_CFD_SimV.txt"

load_h5_and_process_components(h5_file_path, output_file_path)

In [ ]:
#Patient data temporal interpolation (optional)
import numpy as np

def load_txt_data(file_path):
    # Load the data from the txt file, skipping the header
    data = np.loadtxt(file_path, delimiter='\t', skiprows=1)
    timesteps = data[:, 0]  # First column is timestep
    u_values = data[:, 1]  # Second column is u component
    v_values = data[:, 2]  # Third column is v component
    w_values = data[:, 3]  # Fourth column is w component
    return timesteps, u_values, v_values, w_values

def interpolate_values(timesteps_24, u_24, v_24, w_24, timesteps_99):
    # Interpolate actual values to match the 99 timesteps of the predicted data
    u_interpolated = np.interp(timesteps_99, timesteps_24, u_24)
    v_interpolated = np.interp(timesteps_99, timesteps_24, v_24)
    w_interpolated = np.interp(timesteps_99, timesteps_24, w_24)
    return u_interpolated, v_interpolated, w_interpolated

def save_interpolated_data(output_file_path, timesteps_99, u_interpolated, v_interpolated, w_interpolated):
    # Save the interpolated values with the format "Timestep    Transformed u    Transformed v    Transformed w"
    data_to_save = np.column_stack((timesteps_99, u_interpolated, v_interpolated, w_interpolated))
    header = "Timestep\tTransformed u\tTransformed v\tTransformed w"
    np.savetxt(output_file_path, data_to_save, delimiter='\t', header=header, comments='', fmt='%d\t%.12f\t%.12f\t%.12f')

# Path to the txt file
file_path = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_Patient.txt"

# Load the original data
timesteps_24, u_24, v_24, w_24 = load_txt_data(file_path)

# Define the new timesteps (1 to 99 timesteps) (Adjust it for your case)
timesteps_99 = np.arange(1, 100)

# Interpolate the u, v, w values to the new timesteps
u_interpolated, v_interpolated, w_interpolated = interpolate_values(timesteps_24, u_24, v_24, w_24, timesteps_99)

# Save the interpolated values to a new file
output_file_path = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_Patient_inter.txt"
save_interpolated_data(output_file_path, timesteps_99, u_interpolated, v_interpolated, w_interpolated)

print(f"Interpolated data saved to: {output_file_path}")

In [ ]:
#Velocity components plots of patient & cfd data 
import numpy as np
import matplotlib.pyplot as plt

def load_txt_data(file_path):
    """
    Load the velocity data from a tab-delimited text file, skipping the header.
    """
    data = np.loadtxt(file_path, delimiter='\t', skiprows=1)
    timesteps = data[:, 0]  # First column is timesteps
    u_values = data[:, 1]  # Second column is u component
    v_values = data[:, 2]  # Third column is v component
    w_values = data[:, 3]  # Fourth column is w component
    return timesteps, u_values, v_values, w_values

def plot_velocity_comparison(scaled_timesteps_1, data_1, label_1, 
                             scaled_timesteps_2, data_2, label_2, velocity_label):
    """
    Plot the velocity comparison between two datasets for a given component.
    """
    plt.figure(figsize=(10, 6))
    plt.plot(scaled_timesteps_1, data_1, label=label_1, color='blue', linestyle='-', marker='o')
    plt.plot(scaled_timesteps_2, data_2, label=label_2, color='red', linestyle='--', marker='x')
    plt.title(f"{velocity_label} Velocity Comparison")
    plt.xlabel("Time (s)")
    plt.ylabel(f"{velocity_label} Velocity (m/s)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# File paths
file_path_patient = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_Patient.txt"
file_path_cfd = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_CFD_SimV.txt"

# Load data
timesteps_patient, u_patient, v_patient, w_patient = load_txt_data(file_path_patient)
timesteps_cfd, u_cfd, v_cfd, w_cfd = load_txt_data(file_path_cfd)

# Scale timesteps to start at 0 and end at 0.98s (Adjust accordingly)
scaled_timesteps_patient = (timesteps_patient - timesteps_patient[0]) / (timesteps_patient[-1] - timesteps_patient[0]) * 0.98
scaled_timesteps_cfd = (timesteps_cfd - timesteps_cfd[0]) / (timesteps_cfd[-1] - timesteps_cfd[0]) * 0.98

# Plot comparisons for each velocity component
plot_velocity_comparison(scaled_timesteps_patient, u_patient, "Patient u Velocity",
                         scaled_timesteps_cfd, u_cfd, "CFD u Velocity", "u")
plot_velocity_comparison(scaled_timesteps_patient, v_patient, "Patient v Velocity",
                         scaled_timesteps_cfd, v_cfd, "CFD v Velocity", "v")
plot_velocity_comparison(scaled_timesteps_patient, w_patient, "Patient w Velocity",
                         scaled_timesteps_cfd, w_cfd, "CFD w Velocity", "w")

In [ ]:
#Velocity comparison plots (scaled)
import numpy as np
import matplotlib.pyplot as plt

def load_txt_data(file_path):
    """
    Load the velocity data from a tab-delimited text file, skipping the header.
    """
    data = np.loadtxt(file_path, delimiter='\t', skiprows=1)
    timesteps = data[:, 0]  # First column is timesteps
    u_values = data[:, 1]  # Second column is u component
    v_values = data[:, 2]  # Third column is v component
    w_values = data[:, 3]  # Fourth column is w component
    return timesteps, u_values, v_values, w_values

def plot_velocity_comparison(scaled_timesteps_1, data_1, label_1, 
                             scaled_timesteps_2, data_2, label_2, 
                             velocity_label, y_min, y_max):
    """
    Plot the velocity comparison between two datasets for a given component.
    """
    plt.figure(figsize=(10, 6))
    plt.plot(scaled_timesteps_1, data_1, label=label_1, color='blue', linestyle='-', marker='o')
    plt.plot(scaled_timesteps_2, data_2, label=label_2, color='red', linestyle='--', marker='x')
    plt.title(f"{velocity_label} Velocity Comparison")
    plt.xlabel("Time (s)")
    plt.ylabel(f"{velocity_label} Velocity (m/s)")
    plt.ylim(y_min, y_max)  # Apply the same y-axis limits for all plots
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# File paths
file_path_patient = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_Patient.txt"
file_path_cfd = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_CFD_SimV.txt"

# Load data
timesteps_patient, u_patient, v_patient, w_patient = load_txt_data(file_path_patient)
timesteps_cfd, u_cfd, v_cfd, w_cfd = load_txt_data(file_path_cfd)

# Scale timesteps to start at 0 and end at 0.98s (Adjust accordingly)
scaled_timesteps_patient = (timesteps_patient - timesteps_patient[0]) / (timesteps_patient[-1] - timesteps_patient[0]) * 0.98
scaled_timesteps_cfd = (timesteps_cfd - timesteps_cfd[0]) / (timesteps_cfd[-1] - timesteps_cfd[0]) * 0.98

# Determine y-axis limits from w-component
w_min = min(np.min(w_patient), np.min(w_cfd)-0.1)
w_max = max(np.max(w_patient), np.max(w_cfd)+0.6)

# Apply the same y-axis scale to all plots
plot_velocity_comparison(scaled_timesteps_patient, u_patient, "Patient u Velocity",
                         scaled_timesteps_cfd, u_cfd, "CFD u Velocity", "u", w_min, w_max)
plot_velocity_comparison(scaled_timesteps_patient, v_patient, "Patient v Velocity",
                         scaled_timesteps_cfd, v_cfd, "CFD v Velocity", "v", w_min, w_max)
plot_velocity_comparison(scaled_timesteps_patient, w_patient, "Patient w Velocity",
                         scaled_timesteps_cfd, w_cfd, "CFD w Velocity", "w", w_min, w_max)

In [ ]:
#Error between the max & min values of cfd and patient data 
import numpy as np

def load_txt_data(file_path):
    # Load the data from the txt file, skipping the header
    data = np.loadtxt(file_path, delimiter='\t', skiprows=1)
    u_values = data[:, 1]  # Second column is u component
    v_values = data[:, 2]  # Third column is v component
    w_values = data[:, 3]  # Fourth column is w component
    return u_values, v_values, w_values

def calculate_percentage_error(original, predicted):
    # Calculate percentage error
    return (abs(original - predicted) / abs(original)) * 100 if original != 0 else float('inf')

def print_min_max_values_and_errors(u1, v1, w1, u2, v2, w2, label1, label2):
    # Print the max and min values for u, v, and w along with percentage errors
    print(f"Max and Min Values Comparison ({label1} vs {label2}):\n")

    for comp, data1, data2 in zip(['u', 'v', 'w'], [u1, v1, w1], [u2, v2, w2]):
        max1, min1 = np.max(data1), np.min(data1)
        max2, min2 = np.max(data2), np.min(data2)

        max_error = calculate_percentage_error(max1, max2)
        min_error = calculate_percentage_error(min1, min2)

        print(f"  {comp}:")
        print(f"    {label1} - Max: {max1}, Min: {min1}")
        print(f"    {label2} - Max: {max2}, Min: {min2}")
        print(f"    Percentage Error - Max: {max_error:.2f}%, Min: {min_error:.2f}%\n")

# Paths to the txt files
file_path_patient = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_Patient.txt"
file_path_cfd = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_CFD_SimV.txt"

# Load data for Patient file
u_patient, v_patient, w_patient = load_txt_data(file_path_patient)

# Load data for CFD file
u_cfd, v_cfd, w_cfd = load_txt_data(file_path_cfd)

# Print max, min values and percentage errors
print_min_max_values_and_errors(u_patient, v_patient, w_patient, u_cfd, v_cfd, w_cfd, "MRI_data_Patient", "MRI_data_CFD")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_txt_data(file_path):
    """
    Load the velocity data from a tab-delimited text file, skipping the header.
    """
    data = np.loadtxt(file_path, delimiter='\t', skiprows=1)
    timesteps = data[:, 0]  # First column is timesteps
    u_values = data[:, 1]  # Second column is u component
    v_values = data[:, 2]  # Third column is v component
    w_values = data[:, 3]  # Fourth column is w component
    return timesteps, u_values, v_values, w_values

def plot_velocity_comparison(timesteps_list, data_list, labels, velocity_label, y_min, y_max):
    """
    Plot the velocity comparison for a given component across multiple datasets.
    """
    plt.figure(figsize=(10, 6))
    colors = ['blue', 'red', 'green']
    linestyles = ['-', '--', '-.']
    markers = ['o', 'x', 'x']
    
    for i, (timesteps, data, label) in enumerate(zip(timesteps_list, data_list, labels)):
        plt.plot(timesteps, data, label=label, color=colors[i], linestyle=linestyles[i], marker=markers[i])
    
    plt.title(f"{velocity_label} Velocity Comparison")
    plt.xlabel("Time (s)")
    plt.ylabel(f"{velocity_label} Velocity (m/s)")
    plt.ylim(y_min, y_max)  # Apply the same y-axis limits for all plots
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# File paths
file_path_patient = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_Patient.txt"
file_path_cfd_simv = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_CFD_SimV.txt"
file_path_cfd_crimson = r"Your Path\Step_3_Velocity_graph_(CFD&Patient)_extraction\MRI_data_CFD.txt"

# Load data
timesteps_patient, u_patient, v_patient, w_patient = load_txt_data(file_path_patient)
timesteps_cfd_simv, u_cfd_simv, v_cfd_simv, w_cfd_simv = load_txt_data(file_path_cfd_simv)
timesteps_cfd_crimson, u_cfd_crimson, v_cfd_crimson, w_cfd_crimson = load_txt_data(file_path_cfd_crimson)

# Scale timesteps to start at 0 and end at 0.98s (Adjust accordingly)
scaled_timesteps_patient = (timesteps_patient - timesteps_patient[0]) / (timesteps_patient[-1] - timesteps_patient[0]) * 0.98
scaled_timesteps_cfd_simv = (timesteps_cfd_simv - timesteps_cfd_simv[0]) / (timesteps_cfd_simv[-1] - timesteps_cfd_simv[0]) * 0.98
scaled_timesteps_cfd_crimson = (timesteps_cfd_crimson - timesteps_cfd_crimson[0]) / (timesteps_cfd_crimson[-1] - timesteps_cfd_crimson[0]) * 0.98

# Determine y-axis limits
w_min = min(np.min(w_patient), np.min(w_cfd_simv), np.min(w_cfd_crimson)) - 0.1
w_max = max(np.max(w_patient), np.max(w_cfd_simv), np.max(w_cfd_crimson)) + 0.3

# Apply the same y-axis scale to all plots
plot_velocity_comparison([scaled_timesteps_patient, scaled_timesteps_cfd_simv, scaled_timesteps_cfd_crimson],
                         [u_patient, u_cfd_simv, u_cfd_crimson],
                         ["Patient u Velocity", "SimV u Velocity", "Crimson u Velocity"], "u", w_min, w_max)

plot_velocity_comparison([scaled_timesteps_patient, scaled_timesteps_cfd_simv, scaled_timesteps_cfd_crimson],
                         [v_patient, v_cfd_simv, v_cfd_crimson],
                         ["Patient v Velocity", "SimV v Velocity", "Crimson v Velocity"], "v", w_min, w_max)

plot_velocity_comparison([scaled_timesteps_patient, scaled_timesteps_cfd_simv, scaled_timesteps_cfd_crimson],
                         [w_patient, w_cfd_simv, w_cfd_crimson],
                         ["Patient w Velocity", "SimV w Velocity", "Crimson w Velocity"], "w", w_min, w_max)